In [59]:
from transformers import RobertaConfig, RobertaTokenizer, RobertaForSequenceClassification, AutoTokenizer, AdamW
from mtl_model import MTLRobertaForSequenceClassification
from tqdm import tqdm
from torch.utils.data import Dataset
import torch
import json
import random
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import defaultdict
import numpy as np
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler, Dataset
import pandas as pd
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support
from utils import get_base_path
import os
import sys
sys.path.append('../')

# from mtl_dataloader import MTLTasks

In [60]:

class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.targets = labels

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        input_id = self.input_ids[idx]
        attention_mask = self.attention_mask[idx]
        label = self.targets[idx]

        return input_id, attention_mask, label


def few_shot_sample(args, df, sample_strategy, mtl_tasks):
    if sample_strategy == 'mv':
        df_mtl_tasks = load_data(args, 'train', mtl_tasks)
        # sample the data which the mtl was trained with
        mtl_task_sample_size = (
            args.budget - (args.n_fewshot_tasks * args.k_shot)) // len(mtl_tasks.split(","))

        # print(f"Sample size: {mtl_task_sample_size}")

        df_mtl_tasks = df_mtl_tasks.sample(mtl_task_sample_size,
                                           random_state=args.seed).reset_index(drop=True)

        subset_label_0 = df_mtl_tasks[df_mtl_tasks[args.label] == 0]
        subset_label_1 = df_mtl_tasks[df_mtl_tasks[args.label] == 1]
        # Sample half from each subset
        sample_1 = subset_label_1.sample(
            min(len(subset_label_1), args.k_shot // 2), random_state=args.seed)
        sample_0 = subset_label_0.sample(
            args.k_shot - len(sample_1), random_state=args.seed)
        sampled_df = pd.concat([sample_1, sample_0]).reset_index()
        sample_ids = sampled_df['id'].tolist()
        df = df[df['id'].isin(sample_ids)].reset_index(drop=True)
    else:
        raise NotImplementedError
    return df


def load_data(args, split, annotators):
    data_file = f"data/{args.dataset}/{args.label}/annotators/{annotators}/{split}.csv"
    data_file = os.path.join(get_base_path(), data_file)
    df = pd.read_csv(data_file)
    return df


def get_dataset(args, df, split, tokenizer, mtl_tasks=None, sample_strategy='mv'):

    if split == 'train':
        df = few_shot_sample(args, df, sample_strategy, mtl_tasks)

        # df = df.sample(args.k_shot,random_state=args.seed).reset_index(drop=True)
    texts = df[args.text_col].tolist()
    labels = df[args.label].tolist()
    encoded_texts = tokenizer(
        texts, padding=True, truncation=True, return_tensors="pt")
    input_ids = encoded_texts["input_ids"]
    attention_mask = encoded_texts['attention_mask']
    # labels = torch.tensor(labels)
    dataset = CustomDataset(input_ids, attention_mask, labels)

    return dataset

In [61]:
def few_shot_train(args, model, train_dataloader, device, optimizer):
    losses = []
    for batch in train_dataloader:
        input_ids, attention_mask, labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model.forward(input_ids=input_ids,
                                attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
    return model, np.mean(losses)


def evaluate(data_loader, model, device):
    predictions = []
    test_labels = []
    test_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask, labels = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            # TODO set classification head
            outputs = model.forward(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            test_loss += loss.item()

            logits = outputs.logits
            predicted = torch.argmax(logits, dim=1)
            predictions.extend(predicted.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    # import IPython; IPython.embed()
    avg_loss = test_loss / len(data_loader)
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels, predictions, average='binary')
    auc = roc_auc_score(test_labels, predictions)
    return [precision, recall, f1, auc, test_loss]


# Generate and save the evaluation result

In [62]:
class Args:
    def __init__(self, label, text_col, dataset, model_name, k_shot, mtl_task_num, few_shot_task):
        self.label = label
        self.text_col = text_col
        self.dataset = dataset
        self.model_name = model_name
        self.k_shot = k_shot
        self.mtl_task_num = mtl_task_num
        self.few_shot_task = few_shot_task


few_shot_task = "Ann6"

# Create an instance of Args and assign values
args = Args(label='Hate', text_col='text', dataset='brexit', few_shot_task=few_shot_task,
            mtl_task_num=len(few_shot_task),  model_name='roberta-base', k_shot=64)

args.balance_ratio = 0.5
args.train_batch_size = 64
args.predict_batch_size = 16
args.balanced_sampler = True

args.sqrt = False
args.few_shot = True
args.few_shot_sample_strategy = 'mv'
args.budget = 2352
args.n_fewshot_tasks = 3
args.seed = 0
args.lr = 2.0e-05
args.freeze_roberta = False
args.val_result_file_name = 'val_result.json'

In [64]:
from tqdm import tqdm

config = RobertaConfig.from_pretrained(
    args.model_name, num_labels=2)  # add label2id id2label!


path = "../results/roberta-base/brexit/Hate/seed_0/budget_2352/mtl_3"
task_base_result_path = f"../results/roberta-base/brexit/Hate/seed_0/budget_2352/few_shot/strategy_{args.few_shot_sample_strategy}/{args.k_shot}/{args.few_shot_task}"
tokenizer = AutoTokenizer.from_pretrained(args.model_name)


df_train = load_data(args, 'train', annotators=args.few_shot_task)
df_val = load_data(args, 'val', annotators=args.few_shot_task)
df_test = load_data(args, 'test',  annotators=args.few_shot_task)


few_shot_val_dataset = get_dataset(
    args, df_val, split='val', tokenizer=tokenizer)
val_data_loader = DataLoader(few_shot_val_dataset, shuffle=True, batch_size=32)

few_shot_test_dataset = get_dataset(
    args, df_test, split='test', tokenizer=tokenizer)
test_data_loader = DataLoader(
    few_shot_test_dataset, shuffle=True, batch_size=32)

device = torch.device("cuda:3")

for root, dirs, files in os.walk(path,):
    for dir in dirs:
        if dir.startswith('mtl') and not few_shot_task in dir:
            print(dir)
            model_path = os.path.join(path, dir, 'best_model')
            model = RobertaForSequenceClassification.from_pretrained(
                model_path, config=config)

            mtl_tasks = dir.split('_')[1]
            few_shot_dataset = get_dataset(
                args, df_train, split='train', tokenizer=tokenizer,
                mtl_tasks=mtl_tasks, sample_strategy=args.few_shot_sample_strategy)


            train_data_loader = DataLoader(
                few_shot_dataset, shuffle=True, batch_size=64)

            model.to(device)
            optimizer = AdamW(model.parameters(),
                                lr=args.lr, weight_decay=0.01)

            # freeze roberta layers
            if args.freeze_roberta:
                for param in model.roberta.parameters():
                    param.requires_grad = False
            best_val_f1 = float('-inf')
            result_path = os.path.join(
                task_base_result_path, dir)
            best_result = {'precision': 0, 'recall': 0,
                            'f1': 0, 'auc': 0, 'loss': 0}
            print(f"\n running for model on MTL tasks : {mtl_tasks} : \n")
            for epoch in range(30):
                model.train()
                model, train_loss = few_shot_train(
                    args, model, train_data_loader, device, optimizer)

                model.eval()
                precision, recall, f1, auc, val_loss = evaluate(
                    val_data_loader, model, device)

                if f1 > best_val_f1:
                    best_val_f1 = f1
                    best_result['precision'] = precision
                    best_result['recall'] = recall
                    best_result['f1'] = f1
                    best_result['auc'] = auc
                    best_result['loss'] = val_loss
                    model.save_pretrained(f"{result_path}/model")
                    print(
                        f"epoch: {epoch} train_loss: {train_loss} val_loss: {val_loss} val_f1: {f1}")
            print(best_result)

            with open(f"{result_path}/{args.val_result_file_name}", "w") as report_file:
                json.dump(best_result, report_file, indent=4)

            # evaluate on test set
            best_model = RobertaForSequenceClassification.from_pretrained(
                f"{result_path}/model", config=config)
            best_model.to(device)
            test_precision, test_recall, test_f1, test_auc, test_loss = evaluate(
                test_data_loader, best_model, device)
            test_result = {'precision': test_precision, 'recall': test_recall,
                            'f1': test_f1, 'auc': test_auc, 'loss': test_loss}
            with open(f"{result_path}/test_result.json", "w") as report_file:
                json.dump(test_result, report_file, indent=4)

mtl_Ann1,Ann3,Ann5_64


/home/golazizi/2DIS/transformers/src/transformers/optimization.py:423: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



 running for model on MTL tasks : Ann1,Ann3,Ann5 : 

epoch: 0 train_loss: 0.7004538178443909 val_loss: 2.427544116973877 val_f1: 0.5490196078431372
{'precision': 0.7, 'recall': 0.45161290322580644, 'f1': 0.5490196078431372, 'auc': 0.7039086413939252, 'loss': 2.427544116973877}
mtl_Ann2,Ann3,Ann5_64


Some weights of the model checkpoint at ../results/roberta-base/brexit/Hate/seed_0/budget_2352/mtl_3/mtl_Ann2,Ann3,Ann5_64/best_model were not used when initializing RobertaForSequenceClassification: ['classification_heads_dict.Ann3.out_proj.weight', 'classification_heads_dict.Ann2.out_proj.bias', 'classification_heads_dict.Ann5.dense.bias', 'classification_heads_dict.Ann5.out_proj.bias', 'classification_heads_dict.Ann2.out_proj.weight', 'classification_heads_dict.Ann5.dense.weight', 'classification_heads_dict.Ann3.dense.weight', 'classification_heads_dict.Ann5.out_proj.weight', 'classification_heads_dict.Ann2.dense.weight', 'classification_heads_dict.Ann3.dense.bias', 'classification_heads_dict.Ann2.dense.bias', 'classification_heads_dict.Ann3.out_proj.bias']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertF


 running for model on MTL tasks : Ann2,Ann3,Ann5 : 

epoch: 0 train_loss: 0.7400623559951782 val_loss: 1.9919660724699497 val_f1: 0.34782608695652173
epoch: 26 train_loss: 0.024341968819499016 val_loss: 4.7189150005578995 val_f1: 0.3902439024390244
{'precision': 0.8, 'recall': 0.25806451612903225, 'f1': 0.3902439024390244, 'auc': 0.6217329879915234, 'loss': 4.7189150005578995}
mtl_Ann2,Ann3,Ann4_64


...

Some weights of the model checkpoint at ../results/roberta-base/brexit/Hate/seed_0/budget_2352/mtl_3/mtl_Ann3,Ann4,Ann5_64/best_model were not used when initializing RobertaForSequenceClassification: ['classification_heads_dict.Ann4.dense.bias', 'classification_heads_dict.Ann3.out_proj.weight', 'classification_heads_dict.Ann5.dense.bias', 'classification_heads_dict.Ann5.out_proj.bias', 'classification_heads_dict.Ann4.out_proj.bias', 'classification_heads_dict.Ann5.dense.weight', 'classification_heads_dict.Ann3.dense.weight', 'classification_heads_dict.Ann5.out_proj.weight', 'classification_heads_dict.Ann3.dense.bias', 'classification_heads_dict.Ann4.dense.weight', 'classification_heads_dict.Ann4.out_proj.weight', 'classification_heads_dict.Ann3.out_proj.bias']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertF


 running for model on MTL tasks : Ann3,Ann4,Ann5 : 

epoch: 0 train_loss: 0.592285692691803 val_loss: 2.206697329878807 val_f1: 0.3913043478260869
epoch: 4 train_loss: 0.2785983085632324 val_loss: 2.1761266589164734 val_f1: 0.43137254901960786
epoch: 5 train_loss: 0.25198128819465637 val_loss: 2.2319527119398117 val_f1: 0.4444444444444444
epoch: 13 train_loss: 0.026894375681877136 val_loss: 3.8158040940761566 val_f1: 0.4583333333333333
epoch: 18 train_loss: 0.009832356125116348 val_loss: 3.5267157182097435 val_f1: 0.5245901639344263
epoch: 19 train_loss: 0.008784815669059753 val_loss: 4.0394300520420074 val_f1: 0.53125
epoch: 26 train_loss: 0.00468640960752964 val_loss: 5.032592296600342 val_f1: 0.5428571428571428
{'precision': 0.48717948717948717, 'recall': 0.6129032258064516, 'f1': 0.5428571428571428, 'auc': 0.7334589121732987, 'loss': 5.032592296600342}
mtl_Ann2,Ann4,Ann5_64


Some weights of the model checkpoint at ../results/roberta-base/brexit/Hate/seed_0/budget_2352/mtl_3/mtl_Ann2,Ann4,Ann5_64/best_model were not used when initializing RobertaForSequenceClassification: ['classification_heads_dict.Ann4.dense.bias', 'classification_heads_dict.Ann2.out_proj.bias', 'classification_heads_dict.Ann5.dense.bias', 'classification_heads_dict.Ann5.out_proj.bias', 'classification_heads_dict.Ann4.out_proj.bias', 'classification_heads_dict.Ann2.out_proj.weight', 'classification_heads_dict.Ann5.dense.weight', 'classification_heads_dict.Ann5.out_proj.weight', 'classification_heads_dict.Ann2.dense.weight', 'classification_heads_dict.Ann4.dense.weight', 'classification_heads_dict.Ann4.out_proj.weight', 'classification_heads_dict.Ann2.dense.bias']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertF


 running for model on MTL tasks : Ann2,Ann4,Ann5 : 

epoch: 0 train_loss: 0.7209435701370239 val_loss: 2.5053921937942505 val_f1: 0.6
{'precision': 0.5384615384615384, 'recall': 0.6774193548387096, 'f1': 0.6, 'auc': 0.7730162467624204, 'loss': 2.5053921937942505}


: 